In [39]:
import numpy as np
import pandas as pd

from transformers import pipeline
from sklearn.metrics import classification_report
import time


In [40]:
#Electronics Dataset:

fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

n_samples=100

N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=690)

samplesize=n_samples*2
N_rewiews=N_rewiews.append(N_rewiew2)

N_rewiews=N_rewiews.reset_index()
N_rewiews['star_rating'].value_counts()

/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_36032/1428832112.py:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)


1    100
5    100
Name: star_rating, dtype: int64

In [41]:
#Model 12 (siebert)

In [42]:
#https://huggingface.co/siebert/sentiment-roberta-large-english
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")

model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

In [43]:
#Testing the speed using 1 sample

#sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

#t_START = time.time()
#out=sentiment_analysis(N_rewiews['review_body'][51])
#elapsed = time.time() - t_START
#print(elapsed)


In [44]:
from transformers import pipeline
    
#sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")


def sentiment_classify(df_sample, column_name):
    sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")



    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]         
        
        
        if len(text) > 514:
            text = text[:514]
            
        prediction={}
        prediction = sentiment_analysis(text)

        if prediction[0]['label']=='POSITIVE':
            df_sample.loc[i, ("sentiment_analysis")]=5


        elif prediction[0]['label']== 'NEGATIVE':
            df_sample.loc[i, ("sentiment_analysis")]=1
        else: 
            print("Error.")
            
        if i%1000==0:
           print ("\n sentiment_classify:   We are at i=", str(i))    
            
            
    return df_sample
            

In [45]:
t_START = time.time()

sentiment_classify(N_rewiews, 'review_body')




 sentiment_classify:   We are at i= 0


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,sentiment_analysis
0,356277,US,12801243,R323SGZ941TLDE,B000Q9ZHHQ,548944986,GE 180-Watt Portable Car Power Inverter (Disco...,Electronics,1,1,1,N,Y,One Star,Stopped working within a month.,2015-05-09,1.0
1,2525752,US,10696881,R2Z90SWA9FCOU0,B003IWKTMG,419293890,Bellagio-Italia Black Leather Disc Storage Bin...,Electronics,1,15,18,N,Y,"good product, if you can actually get it!",3 times have ordered a 3 pack of binders and b...,2011-10-01,1.0
2,1803102,US,11108600,R2ARER1WA2QGCR,B00CMSCV32,996359254,Total War: Rome 2,Electronics,1,7,10,N,N,Disappointed,Rome 2 was a major disappointment for me. It c...,2013-09-18,1.0
3,1904497,US,25192589,R29NU3XQJ9BATI,B005HB7XXE,360800934,Generic replacement for Sharp XR-30X projector...,Electronics,1,0,1,N,Y,Worked for only a short time,Remember this is only used occasionally in our...,2013-07-09,1.0
4,2037186,US,38437022,R3HCJ54K6SJNJ3,B001MQ6BAY,252530455,AC Adapter Power Supply Charger+Cord for Acer ...,Electronics,1,2,3,N,Y,Will damage your computer!,These chargers will damage your computer due t...,2013-04-03,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2543750,US,11059174,R3TP33HN9SZM39,B00001WRSJ,700672076,Sony MDR-V6 Monitor Series Headphones with CCA...,Electronics,5,1,1,N,Y,Awesome!!!!!,I got these yesterday and these are great!!!!!...,2011-08-27,5.0
196,2196478,US,46752203,R2ZKUGGKUMEJDF,B004GV3FRE,786022145,Aurabeam XL-2100 / XL-2100U / A1606034B UHP TV...,Electronics,5,0,0,N,N,Worked great!!!,Got it super quick which was a plus. I was abl...,2013-01-02,5.0
197,1902104,US,9827414,R1OG7FSOF4XO0I,B007NJ3SIM,50452801,Rokono BASS+ Mini Speaker for iPhone / iPad / ...,Electronics,5,1,1,N,Y,Awesome sound for such a little thing!!!,"I received my Rokono yesterday, and am amazed ...",2013-07-11,5.0
198,1523163,US,52901790,R1T41105ZQ5WNX,B00004Z5KA,184536513,Belkin 12-Feet AC Male to Female Replacement P...,Electronics,5,0,0,N,Y,Super power cord,"Excellent power cord, appears to be long enoug...",2014-02-23,5.0


In [46]:
#N_rewiews['sentiment_analysis'].value_counts()

In [47]:
#Evaluation step:
y_pred= N_rewiews['sentiment_analysis']
y_test=N_rewiews['star_rating']

target_names = ['0 = rating of 1',  '1 = rating of 5'] 
print(classification_report(y_test, y_pred, target_names=target_names, digits=6))



                 precision    recall  f1-score   support

0 = rating of 1   0.970297  0.980000  0.975124       100
1 = rating of 5   0.979798  0.970000  0.974874       100

       accuracy                       0.975000       200
      macro avg   0.975048  0.975000  0.974999       200
   weighted avg   0.975048  0.975000  0.974999       200



In [48]:
elapsed = time.time() - t_START

print("Elapsed Time is:  ", elapsed)


Elapsed Time is:   297.75652027130127


In [49]:
#Model 11 / Ref cell 43 at https://github.com/sophiej-s/MSThesis-I/blob/main/wk15.ipynb

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


#from scipy.spatial import distance
#from scipy.spatial import minkowski_distance
#from scipy.spatial.distance import cosine



from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [51]:
#from transformers import pipeline

#classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)


def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)


    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


        if i%1000==0:
           print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample

In [52]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
       if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews

In [53]:
#DETAILED REVIEW for Roberta

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 

In [54]:
def run_SVC(input_df,input_y):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive

    X_train, X_test, y_train, y_test = train_test_split(input_df, input_y['star_rating'],  random_state=56)
    
    #{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}


    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    clf.score(X_test, y_test)
    y_test.value_counts()
    print(clf.score(X_test, y_test))

    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))

In [55]:
#adding embeddings 
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [56]:
class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])




def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics

In [57]:
import gensim


file_embeddings_fast='crawl-300d-2M.vec'

word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)

300


In [58]:
t_START = time.time()


In [59]:
N_rewiews=text_process2(N_rewiews,'review_headline')




 text_process:   We are at i= 0


In [60]:
emotion_roberta(N_rewiews, 'review_body')

emotion_roberta(N_rewiews, 'review_headline')

emotion_roberta(N_rewiews, 'review_headline_processed')



 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 0


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,356277,US,12801243,R323SGZ941TLDE,B000Q9ZHHQ,548944986,GE 180-Watt Portable Car Power Inverter (Disco...,Electronics,1,1,...,0.858583,0.023295,0.037349,0.007135,0.003841,0.003411,0.024708,0.438367,0.356512,0.166027
1,2525752,US,10696881,R2Z90SWA9FCOU0,B003IWKTMG,419293890,Bellagio-Italia Black Leather Disc Storage Bin...,Electronics,1,15,...,0.203070,0.005562,0.103337,0.006920,0.005248,0.001074,0.128450,0.827927,0.007109,0.023272
2,1803102,US,11108600,R2ARER1WA2QGCR,B00CMSCV32,996359254,Total War: Rome 2,Electronics,1,7,...,0.016507,0.965186,0.003916,0.001313,0.002798,0.000611,0.001661,0.011589,0.977039,0.004988
3,1904497,US,25192589,R29NU3XQJ9BATI,B005HB7XXE,360800934,Generic replacement for Sharp XR-30X projector...,Electronics,1,0,...,0.770284,0.028073,0.036587,0.005710,0.003782,0.006153,0.010380,0.874416,0.032884,0.066676
4,2037186,US,38437022,R3HCJ54K6SJNJ3,B001MQ6BAY,252530455,AC Adapter Power Supply Charger+Cord for Acer ...,Electronics,1,2,...,0.015299,0.021847,0.021691,0.753294,0.006521,0.023929,0.002860,0.065130,0.120445,0.027822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2543750,US,11059174,R3TP33HN9SZM39,B00001WRSJ,700672076,Sony MDR-V6 Monitor Series Headphones with CCA...,Electronics,5,1,...,0.099187,0.013262,0.355641,0.012111,0.044718,0.046024,0.480719,0.032192,0.017261,0.366976
196,2196478,US,46752203,R2ZKUGGKUMEJDF,B004GV3FRE,786022145,Aurabeam XL-2100 / XL-2100U / A1606034B UHP TV...,Electronics,5,0,...,0.234366,0.009764,0.130133,0.010119,0.004739,0.004677,0.341810,0.546018,0.022751,0.069885
197,1902104,US,9827414,R1OG7FSOF4XO0I,B007NJ3SIM,50452801,Rokono BASS+ Mini Speaker for iPhone / iPad / ...,Electronics,5,1,...,0.091635,0.007321,0.272439,0.007835,0.025631,0.022429,0.259024,0.034038,0.007083,0.643961
198,1523163,US,52901790,R1T41105ZQ5WNX,B00004Z5KA,184536513,Belkin 12-Feet AC Male to Female Replacement P...,Electronics,5,0,...,0.823893,0.029313,0.103470,0.023261,0.006720,0.003958,0.021403,0.686197,0.071326,0.187136


In [61]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 


In [62]:
embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")

embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')

In [63]:
#combine the embeddings with the emotions
embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')

In [64]:
#{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
run_SVC(embed_emptions_combined,N_rewiews)


0.9
                 precision    recall  f1-score   support

0 = rating of 1   0.965517  0.875000  0.918033        32
1 = rating of 5   0.809524  0.944444  0.871795        18

       accuracy                       0.900000        50
      macro avg   0.887521  0.909722  0.894914        50
   weighted avg   0.909360  0.900000  0.901387        50

[[28  4]
 [ 1 17]]


In [65]:
elapsed = time.time() - t_START

print("Elapsed Time is:  ", elapsed)




Elapsed Time is:   61.147237062454224
